<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/HW/HW2/HW2_GIRAFFE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# My Basil Garden
#Cloud Root


1. **Setup & Configuration**
2. **Data Layer**
3. **Backend / Business Logic**
4. **UI Components**
5. **Frontend Application**
6. **QA Tests**
7. **Launch**


## 1. Setup & Configuration


In [2]:
# Install all runtime dependencies used by the notebook.
!pip -q install pandas numpy scikit-learn gradio nltk sentence-transformers markdown google-generativeai pillow matplotlib requests beautifulsoup4


In [3]:
import os
import re
import json
import time
import html as html_lib
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import gradio as gr
try:
    import markdown as markdown_lib
except ImportError:
    markdown_lib = None
import PIL.Image

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import PorterStemmer

try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    SentenceTransformer = None
    TRANSFORMERS_AVAILABLE = False

try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
except ImportError:
    genai = None
    GEMINI_AVAILABLE = False


def render_markdown(text: str) -> str:
    if markdown_lib is not None:
        return markdown_lib.markdown(text)
    return html_lib.escape(str(text)).replace("\n", "<br>")


def load_gemini_api_key() -> Optional[str]:
    """Read the API key from Colab Secrets first, then from an environment variable."""
    key = os.getenv("GEMINI_API_KEY", "").strip()

    try:
        from google.colab import userdata
        colab_key = userdata.get("GEMINI_API_KEY")
        if colab_key:
            key = str(colab_key).strip()
    except Exception:
        pass

    return key or None


GEMINI_API_KEY = load_gemini_api_key()

if GEMINI_AVAILABLE and GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    vision_model = genai.GenerativeModel("gemini-2.5-flash")
else:
    vision_model = None

print("Gemini:", "configured" if vision_model else "not configured — local fallback remains available")
print("SentenceTransformers:", "available" if TRANSFORMERS_AVAILABLE else "not available — TF-IDF will be used")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini: not configured — local fallback remains available
SentenceTransformers: available


### API key configuration


## 2. Data Layer


In [4]:
basil_papers = [
       {
        "title": "Chemical Composition, Antioxidant and Antimicrobial Activities of Basil (Ocimum basilicum) Essential Oils Depends on Seasonal Variations",
        "authors": (
            "Abdullah Ijaz Hussain, Farooq Anwar, "
            "Syed Tufail Hussain Sherazi, Roman Przybylski"
        ),
        "journal": "Food Chemistry",
        "year": 2008,
        "doi": "10.1016/j.foodchem.2007.12.010",
        "url": "https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666",
        "abstract": (
            "The study examined basil essential oils collected during summer, "
            "autumn, winter, and spring. Essential-oil yield, chemical "
            "composition, antioxidant activity, and antimicrobial activity "
            "changed significantly between seasons. Linalool was the dominant "
            "compound, while winter samples contained more oxygenated "
            "monoterpenes. The oils inhibited all tested bacterial and fungal "
            "microorganisms."
        )
    },

    {
        "title": "Basil Downy Mildew (Peronospora belbahrii): Discoveries and Challenges Relative to Its Control",
        "authors": (
            "Christian A. Wyenandt, James E. Simon, Robert M. Pyne, "
            "Kathryn Homa, Margaret T. McGrath, Shouan Zhang, "
            "Richard N. Raid, Li-Jun Ma, Robert Wick, Li Guo, "
            "Angela Madeiras"
        ),
        "journal": "Phytopathology",
        "year": 2015,
        "doi": "10.1094/PHYTO-02-15-0032-FI",
        "url": "https://doi.org/10.1094/PHYTO-02-15-0032-FI",
        "abstract": (
            "This review examines basil downy mildew caused by "
            "Peronospora belbahrii, a major threat to sweet basil production. "
            "The disease can spread through contaminated seed and is difficult "
            "to control because many commercial sweet basil varieties lack "
            "genetic resistance. The paper reviews breeding programs, pathogen "
            "biology, cultural management, and conventional and organic "
            "fungicide treatments."
        )
    },

    {
        "title": "Effects of Green Synthesized Zinc and Copper Nano-Fertilizers on the Morphological and Biochemical Attributes of Basil Plant",
        "authors": (
            "Ahmadreza Abbasifar, Fatemeh Shahrabadi, "
            "Babak ValizadehKaji"
        ),
        "journal": "Journal of Plant Nutrition",
        "year": 2020,
        "doi": "10.1080/01904167.2020.1724305",
        "url": "https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305",
        "abstract": (
            "Zinc and copper nanoparticles were produced through green "
            "synthesis using basil extract and were applied to basil plants "
            "at different concentrations. Selected combinations improved "
            "plant growth, chlorophyll and carotenoid concentrations, phenolic "
            "and flavonoid contents, and antioxidant activity. The results "
            "show that the biological effects depend strongly on the "
            "concentrations of both nanoparticles."
        )
    },

    {
        "title": "Alteration in Light Spectra Causes Opposite Responses in Volatile Phenylpropanoids and Terpenoids Compared with Phenolic Acids in Sweet Basil (Ocimum basilicum) Leaves",
        "authors": (
            "Minna Kivimäenpää, Adedayo Mofikoya, "
            "Ahmed M. Abd El-Raheem, Johanna Riikonen, "
            "Riitta Julkunen-Tiitto, Jarmo K. Holopainen"
        ),
        "journal": "Journal of Agricultural and Food Chemistry",
        "year": 2022,
        "doi": "10.1021/acs.jafc.2c03309",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9545148/",
        "abstract": (
            "Sweet basil plants were grown under three different LED light "
            "spectra to examine changes in growth, leaf anatomy, photosynthesis, "
            "and secondary metabolites. Light spectra that increased volatile "
            "terpenoids and phenylpropanoids tended to reduce phenolic acids "
            "and plant growth, while spectra that promoted growth and phenolic "
            "acids reduced several volatile compounds. Glandular trichome "
            "density was associated with terpenoid and eugenol production."
        )
    },
   {
    "title": "First Report of Bacterial Leaf Spot Caused by Pseudomonas cichorii on Thai Basil in Hawaii",
    "authors": (
        "Blaine C. Luiz, Wade P. Heller, Eva Brill, "
        "Brian C. Bushe, Lisa M. Keith"
    ),
    "journal": "Plant Disease",
    "year": 2018,
    "doi": "10.1094/PDIS-06-18-0995-PDN",
    "url": "https://apsjournals.apsnet.org/doi/10.1094/PDIS-06-18-0995-PDN",
    "abstract": (
        "Water-soaked, irregular, gray-to-black leaf spots were observed "
        "on Thai basil plants. The lesions ranged from small spots to larger "
        "necrotic areas near the leaf margins. Laboratory isolation and "
        "pathogenicity testing identified Pseudomonas cichorii as the cause "
        "of the bacterial leaf spot disease."
    )
}
]


# --------------------------------------------------
# Paper-file storage required for the RAG workflow
# --------------------------------------------------
# The papers are saved as a JSON file and loaded back from the file.
# This proves that the RAG system is connected to paper files and not only
# to temporary variables in memory.
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PAPERS_FILE_PATH = DATA_DIR / "basil_papers.json"


def save_papers_to_file(papers: List[Dict[str, Any]], file_path: Path = PAPERS_FILE_PATH) -> str:
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(papers, f, ensure_ascii=False, indent=2)
    return str(file_path)


def load_papers_from_file(file_path: Path = PAPERS_FILE_PATH) -> List[Dict[str, Any]]:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


save_papers_to_file(basil_papers)
basil_papers = load_papers_from_file()
print(f"Loaded {len(basil_papers)} papers from {PAPERS_FILE_PATH}")


Loaded 5 papers from data/basil_papers.json


In [5]:
STOP_WORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am',
    'an', 'and', 'any', 'are', 'aren', 'as', 'at', 'be', 'because',
    'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by',
    'can', 'could', 'couldn', 'did', 'didn', 'do', 'does', 'doesn',
    'doing', 'don', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'hadn', 'has', 'hasn', 'have', 'haven', 'having',
    'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his',
    'how', 'i', 'if', 'in', 'into', 'is', 'isn', 'it', 'its', 'itself',
    'just', 'll', 'm', 'ma', 'me', 'might', 'more', 'most', 'mustn',
    'my', 'myself', 'needn', 'no', 'nor', 'now', 'of', 'off',
    'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves',
    'out', 'over', 'own', 're', 's', 'same', 'shan', 'she', 'should',
    'shouldn', 'so', 'some', 'such', 't', 'than', 'that', 'the',
    'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these',
    'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until',
    'up', 've', 'very', 'was', 'wasn', 'we', 'were', 'weren', 'what',
    'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will',
    'with', 'won', 'would', 'wouldn', 'y', 'you', 'your', 'yours',
    'yourself', 'yourselves'
}

TARGET_TERMS = {
    'terpenoids', 'phenolic', 'spectra', 'trichome', 'nanoparticles',
    'chlorophyll', 'flavonoid', 'antioxidant', 'peronospora', 'mildew',
    'seed', 'fungicide', 'essential', 'antimicrobial', 'linalool',
    'composition', 'pseudomonas', 'spot', 'lesion', 'necrotic'
}

stemmer = PorterStemmer()


def build_target_inverted_index(
    papers: List[Dict[str, Any]],
    target_terms: set[str] = TARGET_TERMS
) -> Dict[str, Dict[str, Any]]:
    """Build the tutorial-style term index: total count and source URLs."""
    index: Dict[str, Dict[str, Any]] = {}
    target_stems = {stemmer.stem(term): term for term in target_terms}

    for paper in papers:
        text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()
        paper_url = paper.get("url", "")

        for word in re.findall(r"\w+", text):
            if word in STOP_WORDS:
                continue

            stemmed_word = stemmer.stem(word)
            if word in target_terms or stemmed_word in target_stems:
                saved_term = word if word in target_terms else target_stems[stemmed_word]
                item = index.setdefault(saved_term, {"term": saved_term,"DocIDs": [],"count": 0})
                item["count"] += 1
                if paper_url and paper_url not in item["DocIDs"]:
                    item["DocIDs"].append(paper.get("url", ""))

    return index


inverted_index = build_target_inverted_index(basil_papers)


print("=== TARGET TERM INDEX ===")
for term in sorted(inverted_index):
    item = inverted_index[term]
    print(f"\nterm: {item['term']}")
    print(f"count: {item['count']}")
    print("DocIDs:")

    for doc_id in item["DocIDs"]:
        print(f"  - {doc_id}")


# --------------------------------------------------
# Firebase DB export layer
# --------------------------------------------------
# If FIREBASE_DB_URL is defined, the notebook uploads the index to Firebase
# Realtime Database using the REST API. If it is not defined, the same payload
# is saved locally so the notebook still runs safely during checking.
FIREBASE_DB_URL = "https://cloudroot-9da6c-default-rtdb.europe-west1.firebasedatabase.app/"
FIREBASE_INDEX_PATH = "basil_target_index"
LOCAL_FIREBASE_EXPORT_PATH = DATA_DIR / "firebase_index_export.json"


def build_firebase_index_payload(index: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    payload = {}
    for term, item in sorted(index.items()):
        safe_key = re.sub(r"[^A-Za-z0-9_-]", "_", term)
        payload[safe_key] = {
            "term": item["term"],
            "count": int(item["count"]),
            "links": list(item["DocIDs"]),
        }
    return payload


def upload_index_to_firebase(index: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    payload = build_firebase_index_payload(index)

    if FIREBASE_DB_URL:
        url = f"{FIREBASE_DB_URL}/{FIREBASE_INDEX_PATH}.json"
        response = requests.put(url, json=payload, timeout=20)
        response.raise_for_status()
        return {
            "status": "uploaded_to_firebase",
            "url": url,
            "terms": len(payload),
        }

    with open(LOCAL_FIREBASE_EXPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return {
        "status": "saved_local_firebase_export",
        "file": str(LOCAL_FIREBASE_EXPORT_PATH),
        "terms": len(payload),
    }


def load_index_from_firebase_or_local() -> Dict[str, Any]:
    if FIREBASE_DB_URL:
        url = f"{FIREBASE_DB_URL}/{FIREBASE_INDEX_PATH}.json"
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        return response.json() or {}

    if LOCAL_FIREBASE_EXPORT_PATH.exists():
        with open(LOCAL_FIREBASE_EXPORT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return build_firebase_index_payload(inverted_index)


firebase_sync_result = upload_index_to_firebase(inverted_index)
print("Firebase index sync:", firebase_sync_result)


=== TARGET TERM INDEX ===

term: antimicrobial
count: 2
DocIDs:
  - https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666

term: antioxidant
count: 3
DocIDs:
  - https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666
  - https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305

term: chlorophyll
count: 1
DocIDs:
  - https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305

term: composition
count: 2
DocIDs:
  - https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666

term: essential
count: 3
DocIDs:
  - https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666

term: flavonoid
count: 1
DocIDs:
  - https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305

term: fungicide
count: 1
DocIDs:
  - https://doi.org/10.1094/PHYTO-02-15-0032-FI

term: lesion
count: 1
DocIDs:
  - https://apsjournals.apsnet.org/doi/10.1094/PDIS-06-18-0995-PDN

term: linalool
count: 1
DocIDs:
  - https://www.sciencedirect.

## 3. Backend / Business Logic


### Query-focused summary without generative AI


In [6]:
class SimpleVectorStore:
    """Small in-memory vector store used by the RAG service."""

    def __init__(self):
        self.documents: List[str] = []
        self.embeddings: List[Any] = []
        self.metadatas: List[Dict[str, Any]] = []
        self.ids: List[str] = []

    def clear(self) -> None:
        self.documents.clear()
        self.embeddings.clear()
        self.metadatas.clear()
        self.ids.clear()

    def add(self, embeddings, documents, metadatas, ids) -> None:
        self.embeddings.extend(np.asarray(embeddings))
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        self.ids.extend(ids)

    def query(self, query_embeddings, n_results: int = 3) -> Dict[str, List[List[Any]]]:
        if not self.embeddings:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        similarities = cosine_similarity(
            np.asarray(query_embeddings),
            np.asarray(self.embeddings)
        )[0]

        result_count = min(max(int(n_results), 1), len(self.documents))
        top_indices = np.argsort(similarities)[::-1][:result_count]

        return {
            "ids": [[self.ids[i] for i in top_indices]],
            "documents": [[self.documents[i] for i in top_indices]],
            "metadatas": [[self.metadatas[i] for i in top_indices]],
            "distances": [[float(1 - similarities[i]) for i in top_indices]],
        }


class EcologicalRAG:
    """
    Retrieval service based on Tutorial 7 concepts:
    TF-IDF embeddings, cosine similarity, top-k retrieval and a template response.

    Gemini remains optional. When it is disabled, the answer is built by selecting
    the sentences from the retrieved abstracts that are most similar to the question.
    """

    def __init__(
        self,
        gemini_api_key: Optional[str] = None,#API_KEY
        use_transformers: bool = True
    ):
        self.collection = SimpleVectorStore()
        self.papers: List[Dict[str, Any]] = []
        self.fitted = False
        self.use_transformers = False
        self.embedding_model = None
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")

        if use_transformers and TRANSFORMERS_AVAILABLE:
            try:
                self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.use_transformers = True
            except Exception as exc:
                print(f"SentenceTransformer fallback to TF-IDF: {exc}")

        self.use_gemini = bool(gemini_api_key and GEMINI_AVAILABLE)
        self.text_model = None

        if self.use_gemini:
            genai.configure(api_key=gemini_api_key)
            self.text_model = genai.GenerativeModel("gemini-2.5-flash")

    @staticmethod
    def preprocess_text(text: str) -> str:
        if not text:
            return ""
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\-\.]", " ", text)
        return text.strip()

    @staticmethod
    def _split_sentences(text: str) -> List[str]:
        """Split an abstract into clean sentences without using a language model."""
        clean_text = re.sub(r"\s+", " ", str(text or "")).strip()
        if not clean_text:
            return []

        sentences = re.split(r"(?<=[.!?])\s+", clean_text)
        return [
            sentence.strip()
            for sentence in sentences
            if len(re.findall(r"\w+", sentence)) >= 4
        ]

    def generate_embeddings(self, texts: List[str]):
        if self.use_transformers:
            return self.embedding_model.encode(texts, show_progress_bar=False)

        if not self.fitted:
            self.tfidf.fit(texts)
            self.fitted = True

        return self.tfidf.transform(texts).toarray()

    def load_papers(self, papers_data: List[Dict[str, Any]]) -> None:
        valid_papers = [p for p in papers_data if p.get("abstract", "").strip()]
        if not valid_papers:
            raise ValueError("No papers with abstracts were supplied.")

        documents = [
            self.preprocess_text(f"{paper.get('title', '')}. {paper.get('abstract', '')}")
            for paper in valid_papers
        ]
        metadatas = [
            {
                "title": paper.get("title", "Unknown"),
                "authors": paper.get("authors", "Unknown"),
                "url": paper.get("url", ""),
            }
            for paper in valid_papers
        ]
        ids = [f"paper_{i}" for i in range(len(valid_papers))]

        self.collection.clear()
        self.papers = list(valid_papers)

        # TF-IDF must be refitted whenever the document collection changes.
        self.fitted = False
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
        embeddings = self.generate_embeddings(documents)

        self.collection.add(
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids,
        )

    def search(self, query: str, n_results: int = 3):
        query_processed = self.preprocess_text(query)
        if not query_processed:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        query_embedding = self.generate_embeddings([query_processed])
        return self.collection.query(query_embedding, n_results=n_results)
    #כככככככככככככ
    def _rank_relevant_sentences(
          self,
          query_text: str,
          relevant_papers: List[Dict[str, Any]],
          max_sentences: int = 4
          ) -> List[Dict[str, Any]]:
          """
          Select sentences according to the number of query words
          that appear in each sentence.
          """

          # Prepare the query words using stop words and stemming
          query_words = {
              stemmer.stem(word)
              for word in re.findall(r"\w+", query_text.lower())
              if word not in STOP_WORDS
          }

          ranked_sentences = []

          # Go over the retrieved papers
          for paper_number, paper in enumerate(relevant_papers, start=1):

              sentences = self._split_sentences(
                  paper.get("abstract", "")
              )

              for sentence in sentences:

                  sentence_words = {
                      stemmer.stem(word)
                      for word in re.findall(r"\w+", sentence.lower())
                      if word not in STOP_WORDS
                  }

                  # Count common words between the query and the sentence
                  score = len(query_words.intersection(sentence_words))

                  if score > 0:
                      ranked_sentences.append({
                          "sentence": sentence,
                          "paper_number": paper_number,
                          "score": score
                      })

          # Sort from the highest score to the lowest
          ranked_sentences.sort(
              key=lambda item: item["score"],
              reverse=True
          )

          # If no matching sentence was found, use the first sentence
          # from the highest-ranked paper
          if not ranked_sentences and relevant_papers:
              first_sentences = self._split_sentences(
                  relevant_papers[0].get("abstract", "")
              )

              if first_sentences:
                  ranked_sentences.append({
                      "sentence": first_sentences[0],
                      "paper_number": 1,
                      "score": 0
                  })

          return ranked_sentences[:max_sentences]


    def _local_grounded_response(
                self,
                query_text: str,
                relevant_papers: List[Dict[str, Any]]
            ) -> str:
            """
            Build a simple answer from sentences found in the retrieved papers.
            """

            if not relevant_papers:
                return (
                    "**Research-based answer**\n\n"
                    "No relevant paper was found."
                )

            selected_sentences = self._rank_relevant_sentences(
                query_text,
                relevant_papers
            )

            lines = [
                "**Research-based answer**",
                "",
                f"**Question:** {query_text}",
                "",
            ]

            used_sources = []

            for item in selected_sentences:
                source_number = item["paper_number"]

                lines.append(
                    f"- {item['sentence']} "
                    f"**[Source {source_number}]**"
                )

                if source_number not in used_sources:
                    used_sources.append(source_number)

            lines.extend(["", "**Sources used:**"])

            for source_number in used_sources:
                paper = relevant_papers[source_number - 1]

                source_line = (
                    f"{source_number}. "
                    f"**{paper.get('title', 'Unknown')}** — "
                    f"{paper.get('authors', 'Unknown')}"
                )

                if paper.get("url"):
                    source_line += (
                        f" ([Open article]({paper['url']}))"
                    )

                lines.append(source_line)

            return "\n".join(lines)

    def query(
                self,
                query_text: str,
                n_results: int = 3
            ) -> Dict[str, Any]:
                """Retrieve relevant papers and create an answer."""

                # Find the most relevant papers
                search_results = self.search(
                    query_text,
                    n_results=n_results
                )
                distances = search_results.get("distances", [[]])[0]

                similarities = [1 - distance for distance in distances]

                MIN_SIMILARITY = 0.20

                relevant_indices = [
                    i for i, similarity in enumerate(similarities)
                    if similarity >= MIN_SIMILARITY
                ]

                if not relevant_indices:
                    return {
                        "response": (
                            "I could not find relevant information in the basil research papers "
                            "for this question."
                        ),
                        "papers_found": 0,
                        "papers": [],
                        "search_results": search_results,
                        "mode": "NO_RELEVANT_RESULTS",
                    }
                relevant_papers = []

                for doc_id in search_results["ids"][0]:
                    paper_index = int(doc_id.split("_")[1])
                    relevant_papers.append(
                        self.papers[paper_index]
                    )

                # The default answer is the local summary
                response_mode = "extractive"
                fallback_reason = None

                # Use Gemini only if it is configured
                if self.use_gemini and self.text_model is not None:
                    context_parts = []

                    for source_number, paper in enumerate(
                        relevant_papers,
                        start=1
                    ):
                        document = search_results["documents"][0][
                            source_number - 1
                        ]

                        context_parts.append(
                            f"[Source {source_number}]\n"
                            f"Title: {paper.get('title', 'Unknown')}\n"
                            f"Authors: {paper.get('authors', 'Unknown')}\n"
                            f"URL: {paper.get('url', '')}\n"
                            f"Research content: {document}"
                        )

                    context_text = "\n\n".join(context_parts)

                    prompt = f"""
                          You are an agriculture and basil-plant research assistant.

                          Answer the question using only the supplied research sources.

                          Rules:
                          - Do not invent information.
                          - Cite claims using [Source 1], [Source 2], and so on.
                          - If the sources do not contain enough information, say so.
                          - Give a clear and concise answer.

                          Question:
                          {query_text}

                          Research sources:
                          {context_text}
                          """

                    try:
                        response = self.text_model.generate_content(
                            prompt,
                            generation_config=genai.types.GenerationConfig(
                                temperature=0.3,
                                max_output_tokens=800,
                            ),
                        )

                        response_text = str(
                            response.text or ""
                        ).strip()

                        if not response_text:
                            raise RuntimeError(
                                "Gemini returned an empty response."
                            )

                        response_mode = "gemini"

                    except Exception as exc:
                        fallback_reason = str(exc)

                        response_text = self._local_grounded_response(
                            query_text,
                            relevant_papers
                        )

                else:
                    fallback_reason = "Gemini is not configured."

                    response_text = self._local_grounded_response(
                        query_text,
                        relevant_papers
                    )

                return {
                    "response": response_text,
                    "response_mode": response_mode,
                    "fallback_reason": fallback_reason,
                    "papers_found": len(relevant_papers),
                    "papers": relevant_papers,
                    "search_results": search_results,
                }
# Keep the article-answering flow non-generative by default.
# The image scanner may still use Gemini independently when a key is configured.
USE_GENERATIVE_AI_FOR_RAG = True

rag_system = EcologicalRAG(
    gemini_api_key=(GEMINI_API_KEY if USE_GENERATIVE_AI_FOR_RAG else None),
    use_transformers=False,
)
rag_system.load_papers(basil_papers)

print(f"RAG service ready with {len(rag_system.papers)} papers.")
print(
    "Article summary mode:",
    "Gemini" if rag_system.use_gemini else "TF-IDF extractive template (no generative AI)",
)


RAG service ready with 5 papers.
Article summary mode: TF-IDF extractive template (no generative AI)


In [7]:
def build_paper_index(paper: Dict[str, Any]) -> Dict[str, int]:
    index: Dict[str, int] = {}
    text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()

    for word in re.findall(r"\w+", text):
        if word in STOP_WORDS:
            continue
        stemmed_word = stemmer.stem(word)
        index[stemmed_word] = index.get(stemmed_word, 0) + 1

    return index


def prepare_query_words(query: str) -> List[str]:
    return [
        stemmer.stem(word)
        for word in re.findall(r"\w+", query.lower())
        if word not in STOP_WORDS and word not in {"and", "or"}
    ]


def boolean_search(query: str, mode: str = "AUTO", limit: int = 3) -> List[Dict[str, Any]]:
    query = query.strip()
    if not query:
        return []

    normalized_mode = mode.upper()
    if normalized_mode == "AUTO":
        lowered = f" {query.lower()} "
        normalized_mode = "AND" if " and " in lowered else "OR"

    clean_query = re.sub(r"\b(?:AND|OR)\b", " ", query, flags=re.IGNORECASE)
    query_words = prepare_query_words(clean_query)
    if not query_words:
        return []

    matched_papers = []

    for paper in basil_papers:
        paper_index = build_paper_index(paper)
        found_words = {
            word: paper_index[word]
            for word in query_words
            if word in paper_index
        }

        is_match = (
            len(found_words) == len(set(query_words))
            if normalized_mode == "AND"
            else bool(found_words)
        )

        if is_match:
            result = dict(paper)
            result["found_words"] = found_words
            result["score"] = sum(found_words.values())
            result["mode"] = normalized_mode
            matched_papers.append(result)

    matched_papers.sort(key=lambda item: item["score"], reverse=True)
    return matched_papers[:limit]


def advanced_html_search(query: str, mode: str = "AUTO"):
    """Generator used by Gradio to show a loading state and then final results."""
    if not query or not query.strip():
        yield "<div style='color:#ef4444; padding:20px;'>Please enter a search term.</div>"
        return

    yield """
    <div style="text-align:center; padding:40px;">
        <div style="font-size:40px; margin-bottom:10px;">🔍</div>
        <h3 style="color:#7edf62;">Scanning Knowledge Base...</h3>
    </div>
    """

    matched_papers = boolean_search(query, mode)


    if not matched_papers:
        papers_html = f"""
        <div style="text-align:center; padding:30px; color:#aabd9e;">
            <h3>No exact Boolean Search matches for “{html_lib.escape(query)}”.</h3>
        </div>
        """
    else:
        papers_html = (
            f"<div style='color:#9cb896; margin-bottom:20px;'>"
            f"Found {len(matched_papers)} Boolean result(s).</div>"
        )

        for paper in matched_papers:
            found_words_text = ", ".join(
                f"{html_lib.escape(word)}: {count}"
                for word, count in paper["found_words"].items()
            )
            papers_html += f"""
            <div style="background:rgba(26,38,20,.4); border:1px solid rgba(74,124,46,.2);
                        border-radius:12px; padding:20px; margin-bottom:16px;">
                <div style="font-size:18px; font-weight:700; color:#c2eaaf;">
                    {html_lib.escape(paper.get("title", "Unknown"))}
                </div>
                <div style="font-size:13px; color:#9cb896; margin:8px 0 12px;">
                    {html_lib.escape(paper.get("authors", "Unknown"))}
                </div>
                <div style="font-size:14px; color:#d1d5db; line-height:1.6;">
                    {html_lib.escape(paper.get("abstract", ""))}
                </div>
                <div style="margin-top:14px; color:#fbbf24; font-size:12px;">
                    Score: {paper["score"]} · Found: {found_words_text}
                </div>
                <div style="margin-top:12px;">
                    <a href="{html_lib.escape(paper.get('url', '#'))}" target="_blank"
                       style="color:#7edf62;">Open source ↗</a>
                </div>
            </div>
            """

    yield """
    <div style="text-align:center; padding:30px; color:#fbbf24;">
        <div style="font-size:32px;">📚 ⏳</div>
        <h3>Building a research-based summary...</h3>
    </div>
    """ + papers_html

    rag_result = rag_system.query(query, n_results=3)
    summary_html = render_markdown(rag_result["response"])

    is_ai_response = rag_result.get("response_mode") == "gemini"
    fallback_was_used = bool(rag_result.get("fallback_reason"))

    if is_ai_response:
        summary_title = "✨ AI-Enhanced Summary"
        summary_note = (
            "Generated with Gemini using the retrieved research papers."
        )

    elif fallback_was_used:
        summary_title = "📚 Research-Based Summary"
        summary_note = (
            "Gemini was unavailable, so the system automatically "
            "created a local summary from the retrieved papers."
        )

    else:
        summary_title = "📚 Research-Based Summary"
        summary_note = (
            "Built locally from the most relevant sentences "
            "in the retrieved papers — no generative AI."
        )

    final_html = f"""
    <div style="background:linear-gradient(135deg, rgba(26,38,20,.8), rgba(15,26,10,.9));
                border:1px solid rgba(245,158,11,.3); border-radius:16px;
                padding:24px; margin-bottom:24px;">
        <h3 style="color:#fbbf24; margin-top:0;">{summary_title}</h3>
        <div style="color:#9cb896; font-size:12px; margin-bottom:14px;">{summary_note}</div>
        <div style="color:#e8f0e4; font-size:15px; line-height:1.7;">{summary_html}</div>
    </div>
    """

    yield final_html + papers_html


def search_and_wrap(query: str):
    for step in advanced_html_search(query):
        yield f"<div class='results-container'>{step}</div>"


# --------------------------------------------------
# Index dashboard helpers: table + chart in one Gradio screen
# --------------------------------------------------
def render_index_dashboard_html() -> str:
    stored_index = load_index_from_firebase_or_local()
    rows = []

    for item in sorted(stored_index.values(), key=lambda x: x.get("count", 0), reverse=True):
        links_html = "<br>".join(
            f"<a href='{html_lib.escape(link)}' target='_blank'>source</a>"
            for link in item.get("links", [])
        ) or "-"

        rows.append(f"""
        <tr>
            <td>{html_lib.escape(str(item.get('term', '')))}</td>
            <td>{int(item.get('count', 0))}</td>
            <td>{links_html}</td>
        </tr>
        """)

    return f"""
    <div class="page-card">
        <div class="page-title">Indexed Terms Dashboard</div>
        <div class="page-subtitle">
            Terms are loaded from Firebase when FIREBASE_DB_URL is configured; otherwise from the local Firebase export file.
        </div>
        <table style="width:100%; border-collapse:collapse; margin-top:18px; color:#e8f0e4;">
            <thead>
                <tr style="color:#7edf62; text-align:left; border-bottom:1px solid rgba(126,223,98,.35);">
                    <th style="padding:8px;">Term</th>
                    <th style="padding:8px;">Count</th>
                    <th style="padding:8px;">Links</th>
                </tr>
            </thead>
            <tbody>{''.join(rows)}</tbody>
        </table>
    </div>
    """


def make_index_count_chart():
    stored_index = load_index_from_firebase_or_local()
    terms = []
    counts = []

    for item in sorted(stored_index.values(), key=lambda x: x.get("count", 0), reverse=True):
        terms.append(str(item.get("term", "")))
        counts.append(int(item.get("count", 0)))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(terms, counts)
    ax.set_title("Target term appearances in basil papers")
    ax.set_xlabel("Term")
    ax.set_ylabel("Appearances")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    return fig


def refresh_index_dashboard():
    return render_index_dashboard_html(), make_index_count_chart()


def rag_question_answer(question: str, n_results: int = 3) -> str:
    if not question or not question.strip():
        return "<div class='page-card'>Please enter a question.</div>"

    result = rag_system.query(question, n_results=int(n_results))
    return f"""
    <div class="page-card">
        <div class="page-title">RAG Answer</div>
        <div class="page-subtitle">Mode: {html_lib.escape(str(result.get('response_mode', result.get('mode', 'unknown'))))}</div>
        <div style="color:#e8f0e4; line-height:1.7; margin-top:14px;">
            {render_markdown(result.get('response', ''))}
        </div>
    </div>
    """


In [8]:
# Image upload and AI scanner service

def get_uploaded_file_path(file_obj: Any) -> Optional[str]:
    if file_obj is None:
        return None
    if isinstance(file_obj, str):
        return file_obj
    for attribute in ("name", "path"):
        value = getattr(file_obj, attribute, None)
        if value:
            return value
    return None


def update_upload_preview(files):
    """Return the upload summary cards and gallery paths for Gradio."""
    if files is None:
        files = []
    elif not isinstance(files, (list, tuple)):
        files = [files]

    image_paths = []
    for file_obj in files:
        path = get_uploaded_file_path(file_obj)
        if path and os.path.exists(path):
            image_paths.append(path)

    images_count = len(image_paths)

    if images_count > 0:
        status_text = "Uploaded Successfully"
        status_color = "#30d158"
        status_icon = "✅"
    else:
        status_text = "No Image Uploaded"
        status_color = "#ffcc00"
        status_icon = "⚠️"

    upload_status_html = f"""
    <div style="display:grid; grid-template-columns:repeat(3, 1fr);
                gap:18px; margin:20px 0 24px 0;">

        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">🖼️</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">
                IMAGES UPLOADED
            </div>
            <div style="color:white; font-size:28px; font-weight:900; margin-top:6px;">
                {images_count}
            </div>
        </div>

        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">{status_icon}</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">
                UPLOAD STATUS
            </div>
            <div style="color:{status_color}; font-size:20px; font-weight:900; margin-top:6px;">
                {status_text}
            </div>
        </div>

        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">🚀</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">
                READY TO ANALYZE
            </div>
            <div style="color:white; font-size:28px; font-weight:900; margin-top:6px;">
                {images_count}
            </div>
        </div>

    </div>
    """

    return upload_status_html, image_paths


def run_gemini_analysis(files):
    files = files or []
    if not files:
        return "<div style='color:#ff453a; padding:10px;'>⚠️ Upload at least one image first.</div>"

    if vision_model is None:
        return (
            "<div class='page-card' style='color:#ffcc00;'>"
            "Gemini image analysis is disabled. Add `GEMINI_API_KEY` to Colab Secrets and rerun Setup."
            "</div>"
        )

    reports = []

    for index, file_obj in enumerate(files, start=1):
        try:
            image_path = get_uploaded_file_path(file_obj)
            if not image_path:
                raise ValueError("The uploaded file path could not be read.")

            image = PIL.Image.open(image_path)
            filename = os.path.basename(image_path)
            prompt = (
                "You are an expert plant pathologist. Analyze this basil image. "
                "Return only valid JSON with keys: "
                "status (healthy, mild, or severe), diagnosis (short string), "
                "and recommendations (list of exactly three actionable strings)."
            )

            response = vision_model.generate_content([prompt, image])
            clean_text = response.text.strip().replace("```json", "").replace("```", "")
            result = json.loads(clean_text)

            status = str(result.get("status", "unknown")).lower()
            diagnosis = html_lib.escape(str(result.get("diagnosis", "No diagnosis provided.")))
            recommendations = result.get("recommendations", [])
            if not isinstance(recommendations, list):
                recommendations = [str(recommendations)]

            status_styles = {
                "healthy": ("✅ Healthy", "#30d158", "rgba(48,209,88,.4)"),
                "mild": ("🟡 Mild issue", "#ffcc00", "rgba(255,204,0,.4)"),
                "severe": ("🔴 Severe issue", "#ff453a", "rgba(255,69,58,.4)"),
            }
            status_label, status_color, border_color = status_styles.get(
                status, ("⚪ Unknown", "#aabd9e", "rgba(170,189,158,.4)")
            )

            recommendations_html = "".join(
                f"<li>{html_lib.escape(str(item))}</li>"
                for item in recommendations[:3]
            )

            reports.append(f"""
            <div class="page-card" style="margin-top:14px; border:1px solid {border_color};">
                <div style="display:flex; justify-content:space-between; gap:12px;">
                    <strong style="color:white;">{html_lib.escape(filename)}</strong>
                    <span style="color:{status_color}; font-weight:800;">{status_label}</span>
                </div>
                <p style="color:#e8f0e4;"><strong>Diagnosis:</strong> {diagnosis}</p>
                <div style="color:#aabd9e;"><strong>Recommendations:</strong>
                    <ul>{recommendations_html}</ul>
                </div>
            </div>
            """)
        except Exception as exc:
            reports.append(
                f"<div style='color:#ff453a; padding:10px;'>"
                f"⚠️ Image #{index} failed: {html_lib.escape(str(exc))}</div>"
            )

    return f"""
    <div class="page-card" style="margin-top:24px;">
        <h3 style="color:#7edf62;">🧬 AI Batch Diagnostics Report</h3>
        {''.join(reports)}
    </div>
    """


In [9]:
# Sensor data service

from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import pandas as pd
import requests
import html as html_lib
from typing import Any, Dict, List, Optional, Tuple

BASE_URL = "https://server-cloud-v645.onrender.com"
SENSORS_CONNECTED = True

SENSOR_SPECS = {
    "temperature": {"icon": "🌡️", "label": "Temperature", "unit": "°C"},
    "humidity": {"icon": "💧", "label": "Air Humidity", "unit": "%"},
    "soil": {"icon": "🌱", "label": "Soil Moisture", "unit": "%"},
}


def get_sensor_data(feed: str, limit: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Read one sensor feed, normalize the server response, and convert to Israel Time."""
    if not SENSORS_CONNECTED:
        return pd.DataFrame(), {
            "error": "Sensors disconnected",
            "feed": feed
        }
    try:
        safe_limit = max(1, min(int(limit), 1000))
        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": safe_limit},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
    except Exception as exc:
        return pd.DataFrame(), {"error": str(exc), "feed": feed}

    if isinstance(payload, list):
        rows = payload
    elif isinstance(payload, dict):
        rows = payload.get("data", [])
    else:
        return pd.DataFrame(), {
            "error": "The server returned an unsupported response format.",
            "feed": feed,
        }

    if not isinstance(rows, list):
        return pd.DataFrame(), {
            "error": "The server response does not contain a data list.",
            "feed": feed,
        }

    df = pd.DataFrame(rows)
    if df.empty:
        return df, payload if isinstance(payload, dict) else {"data": rows}

    if "created_at" not in df.columns:
        for candidate in ("createdAt", "timestamp", "time"):
            if candidate in df.columns:
                df = df.rename(columns={candidate: "created_at"})
                break

    if "value" not in df.columns:
        for candidate in ("feed_value", "sensor_value", "reading"):
            if candidate in df.columns:
                df = df.rename(columns={candidate: "value"})
                break

    if "created_at" in df.columns:
        # Convert to datetime object
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

        # Convert timezone from UTC to Israel Time (Asia/Jerusalem)
        if df["created_at"].dt.tz is None:
            df["created_at"] = df["created_at"].dt.tz_localize('UTC')
        df["created_at"] = df["created_at"].dt.tz_convert('Asia/Jerusalem').dt.tz_localize(None)

        df = df.sort_values("created_at", na_position="first").reset_index(drop=True)

    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df, payload if isinstance(payload, dict) else {"data": rows}


def sensor_status(feed: str, value: Any) -> Tuple[str, str]:
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return "No Data", "#aabd9e"

    if feed == "temperature":
        if numeric_value < 18:
            return "Low", "#4d8dff"
        if numeric_value <= 30:
            return "Optimal", "#30d158"
        return "High", "#ff453a"

    if feed == "humidity":
        if numeric_value < 40:
            return "Low", "#ffcc00"
        if numeric_value <= 70:
            return "Optimal", "#30d158"
        return "High", "#4d8dff"

    if feed == "soil":
        if numeric_value < 25:
            return "Low", "#ff453a"
        if numeric_value <= 80:
            return "Optimal", "#30d158"
        return "Too Wet", "#ffcc00"

    return "Unknown", "#aabd9e"


def check_manager_alerts(df: pd.DataFrame, feed: str) -> str:
    if df.empty or "value" not in df.columns:
        return "⚠️ No valid reading is available."

    values = df["value"].dropna()
    if values.empty:
        return "⚠️ No numeric reading is available."

    latest = float(values.iloc[-1])
    status, _ = sensor_status(feed, latest)

    messages = {
        ("temperature", "Low"): "❄️ Temperature is too low for basil. Move the plant to a warmer area.",
        ("temperature", "Optimal"): "✅ Temperature is within the recommended basil range.",
        ("temperature", "High"): "🔥 Temperature is too high. Add shade and check ventilation.",
        ("humidity", "Low"): "💧 Air humidity is low. Consider gentle misting or a humidifier.",
        ("humidity", "Optimal"): "✅ Air humidity is within the preferred range.",
        ("humidity", "High"): "🌫️ Air humidity is high. Improve airflow to reduce disease risk.",
        ("soil", "Low"): "🚿 Soil moisture is low. Check whether the basil needs watering.",
        ("soil", "Optimal"): "✅ Soil moisture is within the preferred range.",
        ("soil", "Too Wet"): "⚠️ Soil is too wet. Pause watering and check drainage.",
    }
    return messages.get((feed, status), f"Status: {status}")


def _latest_numeric_value(df: pd.DataFrame) -> Optional[float]:
    if df.empty or "value" not in df.columns:
        return None
    values = df["value"].dropna()
    if values.empty:
        return None
    return round(float(values.iloc[-1]), 2)


def get_latest_value_for_card(feed: str, default_value: Any = "--"):
    df, _ = get_sensor_data(feed, 10)
    value = _latest_numeric_value(df)
    return value if value is not None else default_value


def get_sensor_snapshot(limit: int = 10) -> Dict[str, Dict[str, Any]]:
    snapshot: Dict[str, Dict[str, Any]] = {
        feed: {"value": None, "error": None}
        for feed in SENSOR_SPECS
    }

    with ThreadPoolExecutor(max_workers=len(SENSOR_SPECS)) as executor:
        future_to_feed = {
            executor.submit(get_sensor_data, feed, limit): feed
            for feed in SENSOR_SPECS
        }

        for future in as_completed(future_to_feed):
            feed = future_to_feed[future]
            try:
                df, raw_data = future.result()
                value = _latest_numeric_value(df)
                error = None
                if value is None:
                    if isinstance(raw_data, dict):
                        error = raw_data.get("error") or "No readings returned by the server."
                    else:
                        error = "No readings returned by the server."
                snapshot[feed] = {"value": value, "error": error}
            except Exception as exc:
                snapshot[feed] = {"value": None, "error": str(exc)}

    return snapshot


def _placeholder_snapshot() -> Dict[str, Dict[str, Any]]:
    return {
        feed: {"value": None, "error": None}
        for feed in SENSOR_SPECS
    }


def _format_sensor_value(value: Optional[float], unit: str) -> str:
    return "No data" if value is None else f"{value:g}{unit}"


def update_live_sensor_screen(feed: str, limit: int, date_filter: str = ""):
    fetch_limit = 500 if date_filter else limit
    df, raw_data = get_sensor_data(feed, fetch_limit)

    if df.empty or "value" not in df.columns or df["value"].dropna().empty:
        raw_error = raw_data.get("error", raw_data) if isinstance(raw_data, dict) else raw_data
        error_message = html_lib.escape(str(raw_error))
        return (
            f"<div class='page-card' style='color:#ff453a;'>Sensor error: {error_message}</div>",
            None,
            df,
        )

    # Apply Date Filter if provided (Now natively matches Israel Time)
    if "created_at" in df.columns and date_filter:
        df = df[df["created_at"].dt.strftime("%Y-%m-%d") == date_filter.strip()]

    df = df.tail(int(limit))

    if df.empty:
        return (
            f"<div class='page-card' style='color:#ffcc00;'>No data found for the selected date.</div>",
            None,
            df,
        )

    latest_value = float(df["value"].dropna().iloc[-1])
    manager_alert = check_manager_alerts(df, feed)

    html_result = f"""
    <div class="main-wrapper">
        <div class="page-card">
            <div class="page-title">{feed.title()} history</div>
            <div class="page-subtitle">Latest: {latest_value:.2f} · Samples displayed: {len(df)}</div>
            <div style="margin-top:18px; padding:14px; background:rgba(0,0,0,.3);
                        border:1px solid rgba(126,214,98,.2); border-radius:12px;">
                {manager_alert}
            </div>
        </div>
    </div>
    """

    fig, ax = plt.subplots(figsize=(10, 4))
    if "created_at" in df.columns and df["created_at"].notna().any():
        ax.plot(df["created_at"], df["value"], marker="o")
        ax.set_xlabel("Time (Israel)")
    else:
        ax.plot(df.index, df["value"], marker="o")
        ax.set_xlabel("Sample")
    ax.set_ylabel("Value")
    ax.set_title(f"{feed.title()} history")
    plt.xticks(rotation=30)
    plt.tight_layout()

    df_table = df.sort_values("created_at", ascending=False) if "created_at" in df.columns else df.iloc[::-1]

    return html_result, fig, df_table


def make_sensor_cards_html(fetch_live: bool = True):
    snapshot = get_sensor_snapshot() if fetch_live else _placeholder_snapshot()
    cards = []
    for feed, spec in SENSOR_SPECS.items():
        value = snapshot[feed]["value"]
        status, color = sensor_status(feed, value)
        value_text = _format_sensor_value(value, spec["unit"])
        error = snapshot[feed]["error"]
        error_html = (
            f"<div style='color:#ff9f0a; font-size:11px; margin-top:6px;'>"
            f"{html_lib.escape(str(error))}</div>"
            if error else ""
        )
        cards.append(f"""
        <div class="page-card" style="text-align:center;">
            <div style="font-size:30px;">{spec['icon']}</div>
            <div style="font-size:28px; font-weight:900; color:white;">{value_text}</div>
            <div style="font-size:14px; font-weight:800; color:white;">{spec['label']}</div>
            <div style="color:{color}; font-weight:800;">{status}</div>
            {error_html}
        </div>
        """)

    valid_count = sum(item["value"] is not None for item in snapshot.values())
    connection_text = "Live data" if valid_count == 3 else f"{valid_count}/3 feeds available"
    connection_color = "#30d158" if valid_count == 3 else "#ffcc00"

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin-bottom:24px;">
            <div style="display:flex; justify-content:space-between; align-items:center; gap:14px;">
                <div>
                    <div class="page-title">Live Telemetry</div>
                    <div class="page-subtitle">Readings from the Render sensor service</div>
                </div>
                <div style="color:{connection_color}; font-weight:800;">● {connection_text}</div>
            </div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:18px;">
            {''.join(cards)}
        </div>
    </div>
    """


def make_dashboard_sensor_summary_html(fetch_live: bool = True):
    snapshot = get_sensor_snapshot() if fetch_live else _placeholder_snapshot()

    cards = []
    for feed, spec in SENSOR_SPECS.items():
        value = snapshot[feed]["value"]
        status, color = sensor_status(feed, value)
        value_text = _format_sensor_value(value, spec["unit"])
        error = snapshot[feed]["error"]
        details = html_lib.escape(str(error)) if error else status

        cards.append(f"""
        <div class="info-mini-card" style="min-height:105px;">
            <div style="font-size:24px; margin-bottom:6px;">{spec['icon']}</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">{spec['label'].upper()}</div>
            <div style="color:white; font-size:23px; font-weight:900; margin-top:4px;">{value_text}</div>
            <div style="color:{color}; font-size:12px; font-weight:800; margin-top:5px;">{details}</div>
        </div>
        """)

    valid_count = sum(item["value"] is not None for item in snapshot.values())
    if valid_count == 3:
        connection_text = "● All sensor feeds are available"
        connection_color = "#30d158"
    elif valid_count:
        connection_text = f"● Partial data: {valid_count}/3 feeds are available"
        connection_color = "#ffcc00"
    else:
        connection_text = "● Sensor service did not return data"
        connection_color = "#ff453a"

    # Enforce Israel Time for the Last Refresh timestamp
    updated_at = pd.Timestamp.now(tz='Asia/Jerusalem').strftime("%H:%M:%S")

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin:24px 0;">
            <div style="display:flex; justify-content:space-between; align-items:center; gap:12px; flex-wrap:wrap;">
                <div>
                    <div style="font-size:20px; font-weight:900; color:white;">Sensor Summary</div>
                    <div style="color:#aabd9e; font-size:12px; margin-top:4px;">Last refresh: {updated_at} (IL)</div>
                </div>
                <div style="color:{connection_color}; font-weight:800; font-size:13px;">{connection_text}</div>
            </div>
            <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:14px; margin-top:18px;">
                {''.join(cards)}
            </div>
        </div>
    </div>
    """


def refresh_dashboard_sensor_summary():
    return make_dashboard_sensor_summary_html(fetch_live=True)


def refresh_live_sensor_cards():
    return make_sensor_cards_html(fetch_live=True)


def show_sensor_graph(feed: str, limit: int, date_filter: str = ""):
    if not feed:
        feed = "temperature"
    html_result, fig, df = update_live_sensor_screen(feed, limit, date_filter)
    return feed, refresh_live_sensor_cards(), html_result, fig, df

## 4. UI Components


In [10]:
# ==========================================
# CELL 5: Frontend UI & Application Launch
# ==========================================

GLOBAL_CSS = """
/* Clean light design for My Basil Garden - minimal overrides */
:root {
    --bg: #f6f8f4;
    --card: #ffffff;
    --text: #1f2937;
    --muted: #647067;
    --green: #2f7d32;
    --green-soft: #eef7ea;
    --border: #d8e3d1;
    --blue: #2563eb;
}

html, body, gradio-app {
    background: var(--bg) !important;
    color-scheme: light !important;
}

.gradio-container {
    max-width: 1150px !important;
    margin: 0 auto !important;
    padding: 18px !important;
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: Arial, Helvetica, sans-serif !important;
}

.gradio-container * {
    font-family: Arial, Helvetica, sans-serif !important;
    box-sizing: border-box !important;
}

.main-wrapper {
    max-width: 1080px !important;
    margin: 0 auto !important;
    padding: 8px !important;
}

/* Header */
.app-header {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 16px !important;
    padding: 18px 22px !important;
    margin: 8px 0 18px 0 !important;
    box-shadow: 0 3px 12px rgba(31, 41, 55, 0.06) !important;
}
.logo-box {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
}
.logo-icon {
    width: 44px !important;
    height: 44px !important;
    border-radius: 12px !important;
    background: var(--green-soft) !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    font-size: 24px !important;
}
.logo-title {
    color: var(--green) !important;
    font-size: 24px !important;
    font-weight: 800 !important;
    line-height: 1.2 !important;
}
.logo-subtitle {
    color: var(--muted) !important;
    font-size: 14px !important;
    line-height: 1.3 !important;
}

/* Tabs */
.tab-nav button, button[role="tab"] {
    color: var(--muted) !important;
    background: transparent !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    padding: 9px 12px !important;
}
.tab-nav button.selected, button[role="tab"][aria-selected="true"] {
    color: var(--green) !important;
    background: var(--green-soft) !important;
}

/* Cards */
.page-card, .metric-card, .screen-card, .summary-card, .upload-zone,
.reminder-left-card, .reminder-right-card, .manager-reminder-header,
.manager-reminder-form, .task-card-pretty, .reminder-task-card,
.info-mini-card, .search-panel, .live-card, .sensor-card {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 16px !important;
    padding: 18px !important;
    box-shadow: 0 3px 12px rgba(31, 41, 55, 0.05) !important;
}

.page-title, h1, h2, h3 {
    color: var(--green) !important;
    font-weight: 800 !important;
}
.page-subtitle, .subtitle, p, label, span, div {
    color: var(--text);
}

/* Gradio inputs and outputs */
textarea, input, select,
.gr-textbox textarea,
.gr-dropdown,
.gr-number input {
    background: #ffffff !important;
    color: var(--text) !important;
    border-color: var(--border) !important;
}

button {
    border-radius: 10px !important;
    font-weight: 700 !important;
}

/* Tables */
table {
    width: 100% !important;
    border-collapse: collapse !important;
    background: #ffffff !important;
    color: var(--text) !important;
}
th {
    background: var(--green-soft) !important;
    color: var(--green) !important;
    font-weight: 800 !important;
    padding: 10px !important;
    border: 1px solid var(--border) !important;
}
td {
    background: #ffffff !important;
    color: var(--text) !important;
    padding: 10px !important;
    border: 1px solid var(--border) !important;
}

/* Gradio dataframe / AG Grid */
.gr-dataframe, .gr-dataframe *,
.ag-theme-quartz, .ag-theme-quartz *,
.ag-root-wrapper, .ag-root, .ag-header, .ag-header-row,
.ag-header-cell, .ag-cell, .ag-row, .ag-center-cols-container,
.ag-body-viewport, .ag-center-cols-viewport {
    background: #ffffff !important;
    color: var(--text) !important;
    border-color: var(--border) !important;
}
.ag-header, .ag-header-cell {
    background: var(--green-soft) !important;
}
.ag-header-cell-text {
    color: var(--green) !important;
    font-weight: 800 !important;
}
.ag-cell-value, .ag-cell {
    color: var(--text) !important;
}

/* Links and code */
a {
    color: var(--blue) !important;
    font-weight: 700 !important;
}
pre, code, .output-text, .output-text *, .gr-textbox textarea {
    background: #ffffff !important;
    color: var(--text) !important;
}

/* Image upload/gallery */
.gr-image, .gr-file, .small-image-gallery {
    background: #ffffff !important;
    color: var(--text) !important;
}

footer { display: none !important; }

@media (max-width: 700px) {
    .logo-title { font-size: 20px !important; }
    .main-wrapper { padding: 4px !important; }
    .page-card, .screen-card, .summary-card, .upload-zone { padding: 14px !important; }
}
"""

In [11]:
def show_dashboard():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        refresh_dashboard_sensor_summary(),
    )


def show_upload():
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
    )


def show_research():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
    )


def show_live():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        refresh_live_sensor_cards(),
    )


In [12]:
###################
# dashboard_html  #
###################
dashboard_html = """
<div class="main-wrapper">

        <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap:18px; margin-bottom:24px;">

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📷</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">PLANT SCANNER</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Ready</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Image upload available</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📚</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">ACADEMIC SEARCH</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Active</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Boolean search enabled</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📡</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">LIVE TELEMETRY</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Monitoring</div>
            <div style="color:#aabd9e; font-size:12px; margin-top:8px;">Live status is shown below</div>
        </div>

    </div>

</div>
"""


In [13]:
# ==========================================
# MANAGER SETUP /  COMPONENT
# ==========================================

import time
import html as html_lib

def render_task_card(task_data, index):
    task_text = html_lib.escape(str(task_data.get("task", "")))
    task_cost = html_lib.escape(str(task_data.get("cost", "")))

    return f"""
    <div class="reminder-task-card">
        <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:14px;">
            <div>
                <div style="color:#9fc99a; font-size:11px; font-weight:850; text-transform:uppercase; letter-spacing:0.08em; margin-bottom:6px;">
                    Reminder #{index + 1}
                </div>
                <div style="color:#ffffff; font-size:18px; font-weight:850; line-height:1.35;">
                    {task_text}
                </div>
            </div>

            <div style="
                min-width:90px;
                text-align:right;
                background: rgba(0,0,0,0.22);
                border: 1px solid rgba(126,214,98,0.12);
                border-radius: 14px;
                padding: 10px 12px;
            ">
                <div style="color:#8ea88a; font-size:10px; font-weight:850; text-transform:uppercase;">
                    Unit cost
                </div>
                <div style="color:#7edf62; font-size:18px; font-weight:900; margin-top:4px;">
                    ₪{task_cost}
                </div>
            </div>
        </div>
    </div>
    """

def build_manager_setup_section():
    tasks_state = gr.State([])

    with gr.Row(elem_classes=["main-wrapper", "dashboard-split-shell"]):

        # ==========================
        # LEFT SIDE -
        # ==========================
        with gr.Column(scale=1, min_width=420):
            with gr.Group(elem_classes=["reminder-left-card"]):

                gr.HTML("""
                <div style="display:flex; align-items:center; gap:12px; margin-bottom:8px;">
                    <div style="
                        width:50px; height:50px; border-radius:16px;
                        display:flex; align-items:center; justify-content:center;
                        background: rgba(126,214,98,0.12);
                        border: 1px solid rgba(126,214,98,0.22);
                        font-size:26px;
                    ">📝</div>

                    <div>
                        <div style="font-size:22px; font-weight:900; color:white;">
                            Plant Care Tasks
                        </div>
                        <div style="font-size:13px; color:#aabd9e; margin-top:4px;">
                            Add daily maintenance tasks for your basil plant.
                        </div>
                    </div>
                </div>
                """)

                with gr.Row(elem_classes=["reminder-form-row"]):
                    task_input = gr.Textbox(
                        label="Task",
                        placeholder="e.g., Water basil plant",
                        scale=3
                    )

                    unit_cost_input = gr.Number(
                        label="Unit Cost (₪)",
                        value=0,
                        scale=1
                    )

                add_task_btn = gr.Button("➕ Add / Save Task", variant="primary")

                manager_message = gr.HTML("")

                def add_new_task(text, cost, current_tasks):
                    current_tasks = list(current_tasks or [])

                    if not text or not text.strip():
                        return (
                            current_tasks,
                            "",
                            "<div style='color:#ffcc00; font-weight:700; margin-top:8px;'>Please enter a task first.</div>"
                        )

                    try:
                        cost = float(cost)
                    except:
                        cost = 0

                    current_tasks.append({
                        "id": int(time.time() * 1000),
                        "task": text.strip(),
                        "cost": cost
                    })

                    return (
                        current_tasks,
                        "",
                        "<div style='color:#30d158; font-weight:700; margin-top:8px;'>Task added successfully.</div>"
                    )

                add_task_btn.click(
                    fn=add_new_task,
                    inputs=[task_input, unit_cost_input, tasks_state],
                    outputs=[tasks_state, task_input, manager_message]
                )

                @gr.render(inputs=tasks_state)
                def render_dynamic_tasks(tasks_list):
                    if not tasks_list:
                        gr.HTML("""
                        <div style="
                            color:#aabd9e;
                            text-align:center;
                            padding:18px;
                            margin-top:14px;
                            border:1px dashed rgba(126,214,98,0.25);
                            border-radius:14px;
                        ">
                            No tasks yet.
                        </div>
                        """)
                        return

                    for i, task_data in enumerate(tasks_list):
                        gr.HTML(render_task_card(task_data, i))

                        with gr.Row(elem_classes=["reminder-action-row"]):
                            edit_btn = gr.Button("Edit", elem_classes=["reminder-edit-btn"])
                            delete_btn = gr.Button("Delete", elem_classes=["reminder-delete-btn"])

                        def delete_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index < len(current_tasks):
                                current_tasks.pop(index)
                            return (
                                current_tasks,
                                "<div style='color:#30d158; font-weight:700; margin-top:8px;'>Task deleted.</div>"
                            )

                        def edit_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index >= len(current_tasks):
                                return current_tasks, "", 0, ""

                            task_to_edit = current_tasks.pop(index)
                            return (
                                current_tasks,
                                task_to_edit["task"],
                                task_to_edit["cost"],
                                "<div style='color:#ffcc00; font-weight:700; margin-top:8px;'>Edit the task above and click Add / Save Task.</div>"
                            )

                        delete_btn.click(
                            fn=delete_this_task,
                            inputs=[tasks_state],
                            outputs=[tasks_state, manager_message]
                        )

                        edit_btn.click(
                            fn=edit_this_task,
                            inputs=[tasks_state],
                            outputs=[tasks_state, task_input, unit_cost_input, manager_message]
                        )

        # ==========================
        # RIGHT SIDE - INFO PANEL
        # ==========================
        with gr.Column(scale=1, min_width=360):
            with gr.Group(elem_classes=["reminder-right-card"]):
                gr.HTML("""
                <div style="margin-bottom:18px;">
                    <div style="font-size:22px; font-weight:900; color:white; margin-bottom:6px;">
                        About This Dashboard
                    </div>
                    <div style="font-size:14px; color:#aabd9e; line-height:1.6;">
                        This page helps users manage basil plant monitoring in one place.
                        You can review sensor data, upload plant images, search academic information,
                        and maintain daily tasks for plant care.
                    </div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">🌱 What can you track?</div>
                    <div style="font-size:13px; color:#aabd9e;">Temperature, humidity, soil moisture, image uploads, tasks.</div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">📌 Task examples</div>
                    <div style="font-size:13px; color:#aabd9e;">Water basil, check humidity, inspect leaves, clean sensors, or review growth status.</div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">💡 Why is this useful?</div>
                    <div style="font-size:13px; color:#aabd9e;">It helps the user organize routine plant care and estimate maintenance cost over time.</div>
                </div>
                """)


## 5. Frontend Application


In [18]:
def run_qa_tests_for_ui():
    """Run QA tests from the interface and return the printed report."""
    import io
    import contextlib

    qa_func = globals().get("run_qa_tests")
    if qa_func is None:
        return (
            "QA tests are not loaded yet.\n"
            "Please run the QA Tests cell once, then refresh this tab."
        )

    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            qa_func()
        return buffer.getvalue()
    except Exception as error:
        return buffer.getvalue() + f"\n❌ QA failed: {error}"


with gr.Blocks(title="My Basil Garden", css=GLOBAL_CSS) as app:
    gr.HTML("""
    <div class="main-wrapper">
        <div class="app-header">
            <div class="logo-box">
                <div class="logo-icon">🌿</div>
                <div>
                    <div class="logo-title">My Basil Garden</div>
                    <div class="logo-subtitle">Intelligent Plant Monitoring</div>
                    <div class="logo-subtitle">Cloud Root Project</div>
                </div>
            </div>
        </div>
    </div>
    """)

    with gr.Tabs():
        # =====================================================
        # TAB 1: Dashboard
        # =====================================================
        with gr.Tab("🏠 Dashboard"):
            gr.HTML(dashboard_html)
            dashboard_sensor_summary = gr.HTML("""
            <div class="main-wrapper">
                <div class="page-card" style="margin:24px 0; text-align:center; color:#aabd9e;">
                    Loading sensor data...
                </div>
            </div>
            """)
            dashboard_refresh_btn = gr.Button("🔄 Refresh Dashboard Sensors")
            build_manager_setup_section()

        # =====================================================
        # TAB 2: Plant Scanner
        # =====================================================
        with gr.Tab("📷 Plant Scanner"):
            with gr.Column(elem_classes=["main-wrapper"]):
                gr.HTML("""
                <div class="page-card">
                    <div class="page-title">Plant Scanner</div>
                    <div class="page-subtitle">Upload basil images and request an AI diagnosis.</div>
                </div>
                """)

                plant_images = gr.File(
                    label="Upload basil plant images",
                    file_count="multiple",
                    file_types=["image"],
                    elem_classes=["custom-file"],
                )
                upload_status = gr.HTML(update_upload_preview([])[0])
                image_preview = gr.Gallery(
                    label="Uploaded Images Preview",
                    columns=4,
                    rows=1,
                    height=180,
                    object_fit="contain",
                    elem_classes=["small-image-gallery"],
                )
                analyze_btn = gr.Button(
                    "🚀 Run AI Analysis",
                    variant="primary",
                    elem_classes=["custom-run-btn"],
                )
                ai_analysis_output = gr.HTML("")

        # =====================================================
        # TAB 3: Academic Search
        # =====================================================
        with gr.Tab("📚 Academic Search"):
            gr.HTML("""
            <div class="main-wrapper" style="margin-bottom:30px; margin-top:10px;">
                <div style="display:flex; align-items:center; gap:16px;">
                    <div style="font-size:28px;">🔍</div>
                    <div>
                        <div style="font-size:28px; font-weight:800; color:#7edf62;">Academic Search Engine</div>
                        <div style="color:#aabd9e;">Boolean retrieval over basil research papers</div>
                    </div>
                </div>
            </div>
            """)
            with gr.Column(elem_classes=["main-wrapper", "research-content"]):
                with gr.Column(elem_classes=["search-panel"]):
                    search_text = gr.Textbox(
                        placeholder="Examples: mildew OR basil | essential oil AND antimicrobial",
                        show_label=False,
                        lines=4,
                        elem_id="academic-search-input",
                    )

                    search_btn = gr.Button(
                        "Search",
                        variant="primary",
                        elem_id="academic-search-button",
                    )
                search_output = gr.HTML("""
                <div class="page-card" style="text-align:center; padding:60px 20px;">
                    <div style="font-size:64px;">📚</div>
                    <h3 style="color:white;">Ready to explore</h3>
                    <p style="color:#aabd9e;">Search the indexed academic database.</p>
                </div>
                """)

        # =====================================================
        # TAB 4: Index Dashboard
        # =====================================================
        with gr.Tab("📊 Index Dashboard"):
            with gr.Column(elem_classes=["main-wrapper"]):
                gr.HTML("""
                <div class="page-card" style="margin-top:22px;">
                    <div class="page-title">Firebase Index Dashboard</div>
                    <div class="page-subtitle">Indexed terms, source links, and chart from the project index.</div>
                </div>
                """)
                refresh_index_btn = gr.Button("🔄 Refresh Index Dashboard", variant="secondary")
                index_dashboard_output = gr.HTML(render_index_dashboard_html())
                index_chart_output = gr.Plot(value=make_index_count_chart(), label="Term Count Chart")

        # =====================================================
        # TAB 6: Live Telemetry
        # =====================================================
        with gr.Tab("📡 Live Telemetry"):
            sensor_cards = gr.HTML(make_sensor_cards_html(fetch_live=False))

            with gr.Column(elem_classes=["main-wrapper"]):
                with gr.Row():
                    temperature_btn = gr.Button("🌡️ Temperature")
                    humidity_btn = gr.Button("💧 Humidity")
                    soil_btn = gr.Button("🌱 Soil")
                    refresh_cards_btn = gr.Button("🔄 Refresh Cards", variant="primary")

                with gr.Row():
                    sensor_limit = gr.Slider(1, 100, value=10, step=1, label="Samples to display")
                    date_filter = gr.Textbox(label="Filter by Date (YYYY-MM-DD)", placeholder="e.g. 2026-06-09 (Leave empty for live data)")

                active_sensor = gr.State("temperature")

                live_output = gr.HTML("""
                <div class="page-card" style="text-align:center;">
                    <h3 style="color:white;">Choose a sensor</h3>
                    <p style="color:#aabd9e;">The graph and table will appear here.</p>
                </div>
                """)
                live_plot = gr.Plot(label="Sensor Graph")
                live_table = gr.Dataframe(label="Sensor Data Table (Newest first)")


    # Scanner wiring
    plant_images.change(
        update_upload_preview,
        inputs=[plant_images],
        outputs=[upload_status, image_preview],
    )
    analyze_btn.click(
        run_gemini_analysis,
        inputs=[plant_images],
        outputs=[ai_analysis_output],
    )

    # Search wiring
    search_btn.click(search_and_wrap, inputs=[search_text], outputs=[search_output])
    search_text.submit(search_and_wrap, inputs=[search_text], outputs=[search_output])

    # Index dashboard wiring
    refresh_index_btn.click(
        refresh_index_dashboard,
        outputs=[index_dashboard_output, index_chart_output],
    )
    # Sensor Graph update wiring
    temperature_btn.click(
        lambda limit, date: show_sensor_graph("temperature", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )
    humidity_btn.click(
        lambda limit, date: show_sensor_graph("humidity", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )
    soil_btn.click(
        lambda limit, date: show_sensor_graph("soil", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )
    refresh_cards_btn.click(
        refresh_live_sensor_cards,
        outputs=[sensor_cards],
    )

    sensor_limit.change(
        show_sensor_graph,
        inputs=[active_sensor, sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )
    date_filter.change(
        show_sensor_graph,
        inputs=[active_sensor, sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )

    dashboard_refresh_btn.click(
        refresh_dashboard_sensor_summary,
        outputs=[dashboard_sensor_summary],
    )

    qa_btn.click(
        run_qa_tests_for_ui,
        outputs=[qa_output],
    )

    app.load(
        refresh_dashboard_sensor_summary,
        outputs=[dashboard_sensor_summary],
    )

print("Frontend application with a single Gradio cell and tabs constructed successfully.")


/tmp/ipykernel_3283/2344371212.py:22: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="My Basil Garden", css=GLOBAL_CSS) as app:
/tmp/ipykernel_3283/92105817.py:125: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  @gr.render(inputs=tasks_state)


Frontend application with a single Gradio cell and tabs constructed successfully.


## 6. QA Tests


In [15]:
# ==========================================
# QA TESTS
# ==========================================

import tempfile


def run_qa_tests():
    passed_tests = []

    def check(test_name: str, condition: bool):
        """Fail immediately when a test condition is not correct."""
        if not condition:
            raise AssertionError(f"QA test failed: {test_name}")

        passed_tests.append(test_name)

    # ==========================================
    # 1. Papers and inverted index tests
    # ==========================================

    check(
        "Five basil papers are loaded",
        len(basil_papers) == 5
    )

    check(
        "Target inverted index is not empty",
        bool(inverted_index)
    )

    check(
        "Mildew is indexed",
        "mildew" in inverted_index
    )



    # ==========================================
    # 1B. File, Firebase export, and dashboard tests
    # ==========================================

    check(
        "Papers are saved as a JSON file",
        PAPERS_FILE_PATH.exists()
    )

    loaded_papers_from_file = load_papers_from_file()

    check(
        "RAG papers can be loaded from file",
        len(loaded_papers_from_file) == len(basil_papers)
        and loaded_papers_from_file[0]["title"] == basil_papers[0]["title"]
    )

    firebase_payload = build_firebase_index_payload(inverted_index)

    check(
        "Firebase payload contains term counts and links",
        "mildew" in firebase_payload
        and firebase_payload["mildew"]["count"] > 0
        and len(firebase_payload["mildew"]["links"]) > 0
    )

    dashboard_html_test = render_index_dashboard_html()

    check(
        "Index dashboard shows terms and links",
        "Indexed Terms Dashboard" in dashboard_html_test
        and "mildew" in dashboard_html_test.casefold()
        and "source" in dashboard_html_test.casefold()
    )

    # ==========================================
    # 2. Boolean search tests
    # ==========================================

    and_results = boolean_search("mildew AND basil")

    check(
        "Boolean AND returns results",
        len(and_results) > 0
    )

    check(
        "Boolean AND returns only matching papers",
        all(
            {"mildew", "basil"}.issubset(
                set(result["found_words"])
            )
            for result in and_results
        )
    )

    or_results = boolean_search("mildew OR linalool")

    check(
        "Boolean OR returns results",
        len(or_results) > 0
    )

    # ==========================================
    # 3. Local RAG tests
    # ==========================================

    local_rag = EcologicalRAG(
        gemini_api_key=None,
        use_transformers=False
    )

    local_rag.load_papers(basil_papers)

    rag_result = local_rag.query(
        "What causes basil downy mildew?",
        n_results=2
    )

    check(
        "RAG returns two papers",
        rag_result["papers_found"] == 2
    )

    check(
        "RAG response is text",
        isinstance(rag_result["response"], str)
        and len(rag_result["response"].strip()) > 0
    )

    check(
        "RAG uses the non-AI extractive mode",
        rag_result["response_mode"] == "extractive"
    )

    check(
        "RAG retrieves the downy mildew paper",
        any(
            "peronospora belbahrii" in (
                paper.get("title", "")
                + " "
                + paper.get("abstract", "")
            ).casefold()
            for paper in rag_result["papers"]
        )
    )

    check(
        "RAG response includes article sources",
        "Sources used" in rag_result["response"]
        and "Source" in rag_result["response"]
    )

    # ==========================================
    # 4. Sensor status tests
    # ==========================================

    check(
        "37.5°C is correctly marked High",
        sensor_status("temperature", 37.5)[0] == "High"
    )

    check(
        "25°C is correctly marked Optimal",
        sensor_status("temperature", 25)[0] == "Optimal"
    )

    check(
        "100% soil moisture is Too Wet",
        sensor_status("soil", 100)[0] == "Too Wet"
    )

    check(
        "Missing sensor value is No Data",
        sensor_status("humidity", "--")[0] == "No Data"
    )

    # ==========================================
    # 5. Dashboard tests
    # ==========================================

    placeholder_dashboard = make_dashboard_sensor_summary_html(
        fetch_live=False
    )

    normalized_dashboard = placeholder_dashboard.casefold()

    expected_sensor_labels = [
        sensor_spec["label"].casefold()
        for sensor_spec in SENSOR_SPECS.values()
    ]

    check(
        "Dashboard renders all three sensor labels",
        all(
            label in normalized_dashboard
            for label in expected_sensor_labels
        )
    )

    check(
        "Dashboard does not append units to missing values",
        "No data°C" not in placeholder_dashboard
        and "No data%" not in placeholder_dashboard
    )

    # ==========================================
    # 6. Image upload preview tests
    # ==========================================

    empty_status, empty_gallery = update_upload_preview(None)

    check(
        "Empty upload preview is handled",
        "No Image Uploaded" in empty_status
        and "IMAGES UPLOADED" in empty_status
        and empty_gallery == []
    )

    with tempfile.NamedTemporaryFile(suffix=".png") as test_image:
        uploaded_status, uploaded_gallery = update_upload_preview(
            [test_image.name]
        )


        normalized_upload_status = "".join(
            uploaded_status.split()
        )

        check(
            "Successful upload displays the count and status",
            "UploadedSuccessfully" in normalized_upload_status
            and "READYTOANALYZE" in normalized_upload_status
            and ">1<" in normalized_upload_status
            and uploaded_gallery == [test_image.name]
        )


    print(f"✅ {len(passed_tests)} QA tests passed:")

    for test_name in passed_tests:
        print("  -", test_name)


run_qa_tests()

✅ 23 QA tests passed:
  - Five basil papers are loaded
  - Target inverted index is not empty
  - Mildew is indexed
  - Papers are saved as a JSON file
  - RAG papers can be loaded from file
  - Firebase payload contains term counts and links
  - Index dashboard shows terms and links
  - Boolean AND returns results
  - Boolean AND returns only matching papers
  - Boolean OR returns results
  - RAG returns two papers
  - RAG response is text
  - RAG uses the non-AI extractive mode
  - RAG retrieves the downy mildew paper
  - RAG response includes article sources
  - 37.5°C is correctly marked High
  - 25°C is correctly marked Optimal
  - 100% soil moisture is Too Wet
  - Missing sensor value is No Data
  - Dashboard renders all three sensor labels
  - Dashboard does not append units to missing values
  - Empty upload preview is handled
  - Successful upload displays the count and status


## 7. Launch


In [19]:
# Run this cell only after all previous cells finish successfully.
# In Colab, inline=False is usually smoother and avoids scroll lag.
app.launch(share=False, inline=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>